# T1 — Multi-Chart Visualization Notebook · `MEDIUM`

**Task:** Using any real dataset (e.g., Titanic, Iris, or your ETL dataset), create a Jupyter notebook
with at least **6 chart types**: bar, line, histogram, scatter, box plot, and heatmap.  
Each chart must have a proper title, axis labels with units, and a **1-sentence insight caption** beneath it.  
Use **Matplotlib for at least 4** and **Seaborn for at least 2**.

> Dataset: **Titanic** (built into seaborn — no download needed)


In [ ]:
# ── Imports & global dark theme ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Intentional color choices:
#   Teal   (#2a9d8f) = survival / positive / primary series
#   Coral  (#e76f51) = death / negative / contrast series
#   Amber  (#f4a261) = 3rd variable / neutral highlight
#   All three are colorblind-friendly together.
PALETTE = ["#2a9d8f", "#e76f51", "#f4a261", "#264653", "#e9c46a", "#a8dadc"]
BG      = "#0d1117"
TEXT    = "#e6edf3"
GRID    = "#21262d"

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG,
    "axes.edgecolor": GRID, "axes.labelcolor": TEXT,
    "axes.titlecolor": TEXT, "xtick.color": TEXT, "ytick.color": TEXT,
    "text.color": TEXT, "grid.color": GRID, "grid.linestyle": "--",
    "grid.alpha": 0.5, "legend.facecolor": "#161b22",
    "legend.edgecolor": GRID, "font.family": "sans-serif", "font.size": 11,
})
sns.set_theme(style="dark", rc=plt.rcParams)

# Load dataset — seaborn's built-in Titanic; no internet needed after first run
df = sns.load_dataset("titanic")
print(f"Dataset shape: {df.shape}")
print(df[["survived","pclass","age","fare","sex","embarked"]].isnull().sum())


In [ ]:
# ── Chart 1 · BAR — Survival count by passenger class (Matplotlib) ───────────
fig, ax = plt.subplots(figsize=(8, 5))

# Count survivors vs fatalities for each class then place side-by-side bars
survival_by_class = df.groupby(["pclass", "survived"]).size().unstack()
x, width = np.arange(3), 0.35

bars_dead = ax.bar(x - width/2, survival_by_class[0], width,
                   color=PALETTE[1], label="Did not survive", zorder=3)
bars_surv = ax.bar(x + width/2, survival_by_class[1], width,
                   color=PALETTE[0], label="Survived", zorder=3)

# Annotate bar heights so the viewer doesn't need to read the y-axis exactly
for bar in list(bars_dead) + list(bars_surv):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
            str(int(bar.get_height())), ha="center", fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(["1st Class", "2nd Class", "3rd Class"])
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Number of Passengers (count)")
ax.set_title("Chart 1 — Survival Count by Passenger Class")
ax.legend(); ax.yaxis.grid(True, zorder=0)
plt.tight_layout()
plt.savefig("t1_chart1_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Insight: 1st-class had the highest survival rate while 3rd class suffered the most fatalities,")
print("   showing that wealth directly influenced access to lifeboats.")


In [ ]:
# ── Chart 2 · LINE — Average fare over age bins (Matplotlib) ─────────────────
fig, ax = plt.subplots(figsize=(9, 5))

# Bin ages into 5-year intervals, compute mean fare per bin
df_c = df.dropna(subset=["age","fare"]).copy()
df_c["age_bin"] = pd.cut(df_c["age"], bins=range(0, 85, 5))
avg_fare = df_c.groupby("age_bin", observed=True)["fare"].mean()
mid_ages = [iv.mid for iv in avg_fare.index]  # use bin midpoint as x

ax.plot(mid_ages, avg_fare.values, color=PALETTE[0], linewidth=2.5,
        marker="o", markersize=6, markerfacecolor=PALETTE[2])
ax.fill_between(mid_ages, avg_fare.values, alpha=0.15, color=PALETTE[0])

ax.set_xlabel("Age (years, bin midpoint)")
ax.set_ylabel("Average Fare (£ GBP)")
ax.set_title("Chart 2 — Average Fare by Age Bin")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("£%.0f"))
ax.yaxis.grid(True)
plt.tight_layout()
plt.savefig("t1_chart2_line.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Insight: Passengers aged 20–30 paid the lowest average fares;")
print("   affluent older travellers and certain young children skew the curve upward.")


In [ ]:
# ── Chart 3 · HISTOGRAM — Age distribution + KDE overlay (Matplotlib) ────────
fig, ax = plt.subplots(figsize=(9, 5))

ages = df["age"].dropna()

# Main histogram — edgecolor = BG separates bars on the dark background
ax.hist(ages, bins=30, color=PALETTE[0], edgecolor=BG, linewidth=0.6, zorder=3)

# KDE on a twin axis so density scale doesn't distort count axis
ax2 = ax.twinx()
sns.kdeplot(ages, ax=ax2, color=PALETTE[2], linewidth=2.5)
ax2.set_ylabel("Density", color=PALETTE[2])
ax2.tick_params(axis="y", labelcolor=PALETTE[2])
ax2.set_ylim(bottom=0); ax2.yaxis.grid(False)

ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of Passengers (count)")
ax.set_title("Chart 3 — Age Distribution of Titanic Passengers")
ax.yaxis.grid(True, zorder=0)
plt.tight_layout()
plt.savefig("t1_chart3_histogram.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Insight: The distribution peaks sharply at 20–30 years and is right-skewed;")
print("   very few elderly passengers made the voyage.")


In [ ]:
# ── Chart 4 · SCATTER — Age vs Fare coloured by survival (Matplotlib) ────────
fig, ax = plt.subplots(figsize=(9, 6))

# Plot two layers so survivors visually sit on top of fatalities
for survived, label, color in [(0, "Did not survive", PALETTE[1]),
                                (1, "Survived",        PALETTE[0])]:
    s = df[df["survived"] == survived].dropna(subset=["age","fare"])
    ax.scatter(s["age"], s["fare"], c=color, label=label,
               alpha=0.65, s=28, edgecolors="none", zorder=3)

ax.set_xlabel("Age (years)")
ax.set_ylabel("Fare (£ GBP)")
ax.set_title("Chart 4 — Age vs Fare, Coloured by Survival Outcome")
ax.legend()
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("£%.0f"))
ax.yaxis.grid(True, zorder=0); ax.xaxis.grid(True, zorder=0)
plt.tight_layout()
plt.savefig("t1_chart4_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Insight: High-fare passengers cluster heavily among survivors at every age,")
print("   confirming ticket price (a proxy for class) was a strong survival predictor.")


In [ ]:
# ── Chart 5 · BOX PLOT — Fare distribution by class (Seaborn) ────────────────
fig, ax = plt.subplots(figsize=(9, 6))

# Box plot shows median, IQR, whiskers, and outliers in a single view
sns.boxplot(
    data=df, x="pclass", y="fare",
    palette=[PALETTE[0], PALETTE[2], PALETTE[1]],
    order=[1, 2, 3],
    flierprops=dict(marker="o", markersize=3, alpha=0.4,
                    markerfacecolor=TEXT, markeredgecolor="none"),
    linewidth=1.2, ax=ax
)
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Fare (£ GBP)")
ax.set_title("Chart 5 — Fare Distribution by Passenger Class  [Seaborn]")
ax.set_xticklabels(["1st Class", "2nd Class", "3rd Class"])
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("£%.0f"))
ax.yaxis.grid(True, zorder=0)
plt.tight_layout()
plt.savefig("t1_chart5_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Insight: 1st-class fares have the highest median and extreme outliers;")
print("   3rd-class fares are tightly clustered near zero — the wealth gap was enormous.")


In [ ]:
# ── Chart 6 · HEATMAP — Correlation matrix of numeric columns (Seaborn) ──────
fig, ax = plt.subplots(figsize=(8, 6))

# Only select columns that make analytical sense together
numeric_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df[numeric_cols].dropna().corr()

# Diverging palette: negative = red, zero = mid, positive = teal
sns.heatmap(
    corr, annot=True, fmt=".2f",
    cmap=sns.diverging_palette(10, 175, s=90, l=40, as_cmap=True),
    center=0, vmin=-1, vmax=1,
    linewidths=0.4, linecolor=BG,
    annot_kws={"size": 10}, ax=ax
)
ax.set_title("Chart 6 — Numeric Feature Correlation Matrix  [Seaborn]")
plt.tight_layout()
plt.savefig("t1_chart6_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Insight: 'pclass' has the strongest negative correlation with survival while 'fare' is positive;")
print("   age has a weak but consistent negative effect on survival probability.")
